# Avance 3 — Baseline tabular y conjunto ganador

Cuaderno concentrador que reúne los resultados de las libretas anteriores y produce el **conjunto de características ganador** que alimenta a los modelos densos siguientes (U-Net, U-TAE, TSViT, Swin-UNETR) y a los ensambles (voting, bagging, stacking, blending).

Estructura:

1. Comparativa de los 3 modelos (RandomForest, XGBoost, LightGBM) desde `model_comparison_04.parquet`.
2. Resumen de la ablación completa desde `reports/baseline/05_reencuadre/ablation_table.parquet`.
3. Decisión por bloque opcional (FarSLIP, pheno_text, spectral_signature) vía `select_winning_features`.
4. Persistencia del parquet ganador y de un manifiesto JSON con la lista nominal de columnas, de modo que los modelos siguientes consuman exactamente el mismo conjunto.

In [ ]:
COMPARISON_PATH_04 = "reports/baseline/04_baseline/model_comparison_04.parquet"
ABLATION_PATH_05 = "reports/baseline/05_reencuadre/ablation_table.parquet"
FUSED_PATH = "data/features/features_fused_italy.parquet"
WINNING_OUTPUT = "data/features/features_fused_winning_italy.parquet"
FIGURES_SUBDIR = "us-023-preview/Avance3"
REPORTS_SUBDIR = "baseline/Avance3"
PROMOTE_THRESHOLD = 0.005


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Comparativa de los 3 modelos baseline

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from ml.eval.reencuadre_plots import (
    plot_model_comparison_v2_with_v1_overlay,
    plot_optional_blocks_ablation,
)
from ml.eval.feature_ablation import FeatureAblationResult

comparison_path = Path(COMPARISON_PATH_04)
if comparison_path.exists():
    comparison = pl.read_parquet(comparison_path)
    display(Markdown(f'**Comparativa de modelos** (`{comparison_path}`):'))
    display(comparison)
    v2_metrics = {row['model']: row['f1_macro'] for row in comparison.iter_rows(named=True)}
    v1_metrics = {'xgb': 0.41, 'rf': 0.39}  # referencias previas
    fig = plot_model_comparison_v2_with_v1_overlay(v2_metrics, v1_metrics=v1_metrics)
    fig.savefig(env.figures_dir / 'model_comparison_v2.png', bbox_inches='tight')
    display(fig)
    plt.close(fig)
else:
    raise FileNotFoundError(
        f'No existe `{comparison_path}`. Ejecuta `04_baseline.ipynb` antes de este notebook.'
    )


## Ablación completa (FarSLIP, pheno_text y firma espectral)

In [ ]:
ablation_path = Path(ABLATION_PATH_05)
if not ablation_path.exists():
    raise FileNotFoundError(
        f'No existe `{ablation_path}`. Ejecuta `05_reencuadre_fenologico.ipynb` antes.'
    )
ablation_table = pl.read_parquet(ablation_path)
display(ablation_table)

results = [
    FeatureAblationResult(
        feature_set=row['feature_set'],
        model_kind=row['model'],
        f1_macro=row['f1_macro'] if row['f1_macro'] is not None else float('nan'),
        f1_weighted=row['f1_weighted'] if row['f1_weighted'] is not None else float('nan'),
        miou=row['miou'] if row['miou'] is not None else float('nan'),
        n_features=row['n_features'],
        delta_vs_full=row['delta_vs_full'] if row['delta_vs_full'] is not None else float('nan'),
    )
    for row in ablation_table.iter_rows(named=True)
]
fig = plot_optional_blocks_ablation(results)
fig.savefig(env.figures_dir / 'optional_blocks.png', bbox_inches='tight')
display(fig)
plt.close(fig)


## Selección del conjunto ganador con `select_winning_features`

Promovemos cada bloque opcional cuando su `delta_vs_full >= +0.005`. El conjunto base obligatorio incluye AlphaEarth, índices espectrales, atributos fenológicos, ERA5 y SRTM. Las columnas `geom_*` se descartan siempre por mostrar fuga espacial (resultado validado en la ablación).

In [ ]:
from ml.features.winning_features import (
    select_winning_features,
    persist_winning_features,
)

fused_path = Path(FUSED_PATH)
if not fused_path.exists():
    raise FileNotFoundError(
        f'No existe `{fused_path}`. Ejecuta `05_reencuadre_fenologico.ipynb` '
        '(genera el fused completo durante la materializacion).'
    )
fused = pl.read_parquet(fused_path)

winning = select_winning_features(
    ablation_table,
    available_cols=fused.columns,
    promote_threshold=PROMOTE_THRESHOLD,
    discard_geom=True,
)
display(Markdown('**Decisiones por bloque**:'))
display(pl.DataFrame({
    'bloque': list(winning.decisions.keys()),
    'promovido': list(winning.decisions.values()),
}))
display(Markdown(f'**Conjunto ganador**: `{winning.name}` con `{len(winning.feature_cols)}` columnas.'))
display(Markdown('### Rationale'))
display(Markdown(winning.rationale))

winning_path = persist_winning_features(
    winning,
    fused,
    output_path=WINNING_OUTPUT,
    overwrite=True,
)
display(Markdown(f'**Conjunto ganador guardado**: `{winning_path.relative_to(env.repo)}`'))
display(Markdown(f'**Manifiesto JSON**: `{winning_path.with_suffix(".manifest.json").relative_to(env.repo)}`'))


## Nombres de las características ganadoras

Para que los modelos siguientes (densos y ensambles) lean exactamente las mismas columnas, publicamos la **lista nominal exacta** de las características ganadoras. Cualquier modelo posterior que cargue `features_fused_winning_italy.parquet` reúsa esta lista sin tener que reinventar la selección.

In [ ]:
import json
manifest = json.loads(
    Path(WINNING_OUTPUT).with_suffix('.manifest.json').read_text(encoding='utf-8')
)
display(Markdown(f'**Número de características**: `{manifest["n_features"]}`'))
display(Markdown('**Columnas de metadata** (no son características):'))
display(pl.Series('meta_cols', manifest['meta_cols']).to_frame())
display(Markdown('**Características ganadoras** (primeras 40):'))
display(pl.Series('feature', manifest['feature_cols'][:40]).to_frame())
display(Markdown(f'**Total de características**: `{len(manifest["feature_cols"])}`'))


## Conclusiones — cierre del baseline tabular

Con este cuaderno cerramos el baseline tabular:

- Tres modelos (RandomForest, XGBoost, LightGBM) entrenados con validación cruzada espacial de 5 particiones y buffer de 1 km, registrados en MLflow y serializados con joblib.
- Ablación de 8 a 10 conjuntos de características con decisiones documentadas por cada bloque opcional.
- Conjunto de características ganador nombrado y guardado en `features_fused_winning_italy.parquet` junto con un manifiesto JSON que lista cada columna.

## Lo que sigue

Las libretas siguientes (`05_alt_models.ipynb` y `06_final_gemma4_ensembles.ipynb`) cargarán **el mismo parquet ganador** y entrenarán U-Net, U-TAE, TSViT, Swin-UNETR, Gemma 4 26B-MoE con LoRA y los cuatro ensambles (voting, bagging, stacking, blending), garantizando que todos operan sobre el mismo conjunto de características.